# 03 - Evaluation：MLflow GenAI 評估框架

本筆記本示範如何使用 MLflow GenAI evaluate 進行評估，
包含 rule-based scorers、LLM Judge（內部 LLMService）與 LangGraph Workflow 評估。

> 所有 LLM 評估均使用內部 LLMService，不依賴外部 LLM API key。

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

import mlflow
from app.utils.config import init_config
from app.logger import init_mlflow

cfg = init_config()
init_mlflow(cfg)

print("設定與 MLflow 初始化完成。")

## 準備評估資料

```python
{
    "inputs": {"question": "..."},
    "expectations": {"expected_facts": [...], "keywords": "..."},
}
```

In [ ]:
eval_data = [
    {
        "inputs": {"question": "台灣的首都是哪裡？"},
        "expectations": {
            "expected_facts": ["台北"],
            "expected": "台北",
            "keywords": "台北,首都",
        },
    },
    {
        "inputs": {"question": "什麼是 Python？"},
        "expectations": {
            "expected_facts": ["程式語言", "高階語言"],
            "expected": "Python 是一種高階程式語言",
            "keywords": "程式語言,Python",
        },
    },
    {
        "inputs": {"question": "1 + 1 = ?"},
        "expectations": {
            "expected_facts": ["2"],
            "expected": "2",
            "keywords": "2",
        },
    },
]

MOCK_ANSWERS = {
    "台灣的首都是哪裡？": "台灣的首都是台北。",
    "什麼是 Python？": "Python 是一種高階程式語言，廣泛用於資料科學、Web 開發等領域。",
    "1 + 1 = ?": "2",
}

print(f"準備了 {len(eval_data)} 筆評估資料。")

## 方式一：簡單函式作為 predict_fn

`predict_fn` 接受 `inputs` 中的 key-value 作為參數，回傳字串。

In [ ]:
from app.evaluator import run_evaluation
from app.evaluator.scorers import (
    response_not_empty,
    response_length_check,
    exact_match,
    contains_keywords,
)


@mlflow.trace
def my_app(question: str) -> str:
    """模擬 LLM 應用。"""
    return MOCK_ANSWERS.get(question, "我不知道答案。")


results = run_evaluation(
    predict_fn=my_app,
    eval_data=eval_data,
    scorers=[response_not_empty, response_length_check, exact_match, contains_keywords],
    run_name="rule_based_scorers_demo",
)

print("=== Rule-based Scorer 評估結果 ===")
print(results.tables["eval_results"])

## LLM Judge（使用內部 LLMService）

| Judge | 用途 |
|-------|------|
| `create_correctness_judge()` | 檢查是否包含預期事實 |
| `create_relevance_judge()` | 回答與問題的相關性 |
| `create_quality_judge()` | 回答品質評估 |
| `create_tone_judge()` | 專業語調檢查 |
| `create_safety_judge()` | 安全性檢查 |

In [ ]:
from app.evaluator.scorers import (
    create_correctness_judge,
    create_relevance_judge,
    create_quality_judge,
)

try:
    results = run_evaluation(
        predict_fn=my_app,
        eval_data=eval_data,
        scorers=[create_correctness_judge(), create_relevance_judge(), create_quality_judge()],
        run_name="llm_judge_demo",
    )
    print("=== LLM Judge 評估結果 ===")
    print(results.tables["eval_results"])
except Exception as e:
    print(f"[提示] LLM Judge 需要可用的 LLM 服務：{e}")

## 方式二：LangGraph Workflow 作為 predict_fn

使用 `make_workflow_predict_fn()` 將 LangGraph compiled graph 轉為 predict_fn，
自動將 eval_data 中的 inputs 注入 state 並取得回應。

```
eval_data inputs → state_builder → graph.invoke(state) → output_parser → 字串
```

In [ ]:
from langgraph.graph import END, StateGraph, MessagesState
from langchain_core.messages import AIMessage, HumanMessage
from app.evaluator import make_workflow_predict_fn


def mock_llm_node(state: dict) -> dict:
    """從 messages 取得 user message，回傳 mock 回答。"""
    messages = state.get("messages", [])
    user_text = ""
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage):
            user_text = msg.content
            break
        elif isinstance(msg, tuple) and msg[0] == "user":
            user_text = msg[1]
            break
    answer = MOCK_ANSWERS.get(user_text, "我不知道答案。")
    return {"messages": [AIMessage(content=answer)]}


graph = StateGraph(MessagesState)
graph.add_node("llm", mock_llm_node)
graph.set_entry_point("llm")
graph.add_edge("llm", END)
compiled_graph = graph.compile()

# 預設用法：inputs["question"] → MessagesState → 取最後 AI message
predict_fn = make_workflow_predict_fn(compiled_graph)

results = run_evaluation(
    predict_fn=predict_fn,
    eval_data=eval_data,
    scorers=[response_not_empty, exact_match, contains_keywords],
    run_name="workflow_eval_demo",
)

print("=== Workflow 評估結果 ===")
print(results.tables["eval_results"])

### 自訂 state_builder 與 output_parser

如果 workflow 使用自訂 state 欄位，可以自訂轉換邏輯：

In [ ]:
class QAState(MessagesState):
    summary: str = ""


def qa_node(state: dict) -> dict:
    messages = state.get("messages", [])
    user_text = ""
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage):
            user_text = msg.content
            break
        elif isinstance(msg, tuple) and msg[0] == "user":
            user_text = msg[1]
            break
    answer = MOCK_ANSWERS.get(user_text, "我不知道。")
    return {
        "messages": [AIMessage(content=answer)],
        "summary": f"摘要：{answer[:20]}",
    }


g = StateGraph(QAState)
g.add_node("qa", qa_node)
g.set_entry_point("qa")
g.add_edge("qa", END)
custom_graph = g.compile()

predict_fn = make_workflow_predict_fn(
    custom_graph,
    state_builder=lambda question, **kw: {"messages": [("user", question)]},
    output_parser=lambda state: state["summary"],
)

results = run_evaluation(
    predict_fn=predict_fn,
    eval_data=eval_data,
    scorers=[response_not_empty],
    run_name="custom_workflow_eval_demo",
)

print("=== 自訂 Workflow 評估結果 ===")
print(results.tables["eval_results"])

## 方式三：使用真實 LLM 的 Workflow 評估

將 mock 替換為真實的 `LLMService.call_llm()`，搭配 LLM Judge 一起評估。

In [ ]:
from llm_service import LLMService

service = LLMService()


def real_llm_node(state: dict) -> dict:
    messages = state.get("messages", [])
    user_text = ""
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage):
            user_text = msg.content
            break
        elif isinstance(msg, tuple) and msg[0] == "user":
            user_text = msg[1]
            break
    response = service.call_llm(
        user_prompt=user_text,
        system_prompt="你是一個有用的中文助手，請簡潔回答。",
    )
    return {"messages": [AIMessage(content=response.content)]}


real_graph = StateGraph(MessagesState)
real_graph.add_node("llm", real_llm_node)
real_graph.set_entry_point("llm")
real_graph.add_edge("llm", END)
real_compiled = real_graph.compile()

predict_fn = make_workflow_predict_fn(real_compiled)

try:
    results = run_evaluation(
        predict_fn=predict_fn,
        eval_data=eval_data,
        scorers=[response_not_empty, contains_keywords, create_quality_judge()],
        run_name="real_workflow_eval_demo",
    )
    print("=== 真實 LLM Workflow 評估結果 ===")
    print(results.tables["eval_results"])
except Exception as e:
    print(f"[提示] 需要可用的 LLM 服務：{e}")

## Trace Evaluation（對已有結果評估）

如果已有預測結果，可以直接評估而不需要 predict_fn。

In [ ]:
from app.evaluator import run_trace_evaluation

existing_data = [
    {
        "inputs": {"question": "台灣的首都是哪裡？"},
        "outputs": "台灣的首都是台北。",
        "expectations": {"expected": "台北", "keywords": "台北"},
    },
    {
        "inputs": {"question": "1 + 1 = ?"},
        "outputs": "答案是 2。",
        "expectations": {"expected": "2", "keywords": "2"},
    },
]

results = run_trace_evaluation(
    data=existing_data,
    scorers=[response_not_empty, contains_keywords],
    run_name="trace_eval_demo",
)

print("=== Trace Evaluation 結果 ===")
print(results.tables["eval_results"])

## 小結

| 功能 | API | 說明 |
|------|-----|------|
| 執行評估 | `run_evaluation()` | 自動執行 predict_fn + scorers |
| Workflow 評估 | `make_workflow_predict_fn()` | 將 LangGraph graph 轉為 predict_fn |
| Rule-based | `@scorer` decorator | 不需 LLM，純邏輯評分 |
| LLM Judge | `create_*_judge()` | 使用內部 LLMService 的 LLM 評判 |
| Trace 評估 | `run_trace_evaluation()` | 對已有結果評估 |

> 所有 LLM Judge 均使用內部 LLMService，不依賴外部 LLM API。